In [47]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/saurabhshahane/twitter-sentiment-dataset")

Skipping, found downloaded files in ".\twitter-sentiment-dataset" (use force=True to force download)


In [48]:
import pandas as pd

In [49]:
df = pd.read_csv("twitter-sentiment-dataset/Twitter_Data.csv")
df.head()


,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [50]:
df["category"].isnull().sum()  # check for missing labels

np.int64(7)

In [51]:
dropna = df.dropna(subset=["category"])
dropna["category"].isnull().sum()  # confirm no missing labels

np.int64(0)

In [52]:
df["category"].unique()

array([-1.,  0.,  1., nan])

In [53]:
# convert numeric safely
df["category"] = pd.to_numeric(df["category"], errors="coerce")

# remove missing labels
df = df.dropna(subset=["category"])

# convert label type to int
df["category"] = df["category"].astype(int)
print(df["category"].unique())
print(df["category"].isnull().sum())

[-1  0  1]
0


In [54]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

In [55]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [56]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

In [57]:
df["clean_text"] = df["clean_text"].fillna("").apply(preprocess)
df = df[df["clean_text"].str.strip() != ""]

In [58]:
texts = df["clean_text"]
vocab_size = 8000
tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
print(type(sequences))
print(type(sequences[0]))
print(sequences[:5])


<class 'list'>
<class 'list'>
[[1, 242, 636, 27, 1530, 716, 1005, 1228, 1078, 50, 103, 42, 12, 24, 959, 103, 400, 3563, 4865, 1085], [163, 993, 625, 664, 9, 1], [14, 9, 1, 813, 3, 421, 21, 370, 2054, 1, 45, 1, 3528], [314, 200, 4019, 59, 110, 1, 73, 903, 3603, 184, 395, 5712, 3269, 7814, 40, 689, 255, 699], [245, 722, 833, 64, 32, 60, 728, 3355, 1, 77]]


In [59]:
max_length = 100

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [60]:
# clean labels properly
df["category"] = pd.to_numeric(df["category"], errors="coerce")

df = df.dropna(subset=["category"])

df["category"] = df["category"].astype(int)

# ⭐ REQUIRED FIX
df["category"] = df["category"] + 1



In [61]:
# normalize labels to start from 0
df["category"] = df["category"] - df["category"].min()

In [62]:
print(df["category"].unique())

[0 1 2]


In [63]:
y = df["category"].values

In [68]:
df["clean_text"].str.split().apply(len).describe()

count    162893.000000
mean         14.157287
std           7.607589
min           1.000000
25%           8.000000
50%          13.000000
75%          20.000000
max          43.000000
Name: clean_text, dtype: float64

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print(set(y_train))

{np.int64(0), np.int64(1), np.int64(2)}


In [69]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

model = Sequential([
    Embedding(vocab_size, 128),

    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(3, activation='softmax')
])


In [70]:
from tensorflow.keras.optimizers import Adam

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [71]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
    
)

Epoch 1/5
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 604s 147ms/step - accuracy: 0.8643 - loss: 0.4051 - val_accuracy: 0.9030 - val_loss: 0.3124
Epoch 2/5
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 661s 162ms/step - accuracy: 0.9081 - loss: 0.2968 - val_accuracy: 0.9073 - val_loss: 0.2992
Epoch 3/5
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 619s 152ms/step - accuracy: 0.9161 - loss: 0.2606 - val_accuracy: 0.9071 - val_loss: 0.3101
Epoch 4/5
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 628s 154ms/step - accuracy: 0.9256 - loss: 0.2219 - val_accuracy: 0.9043 - val_loss: 0.3462
Epoch 5/5
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 646s 159ms/step - accuracy: 0.9366 - loss: 0.1855 - val_accuracy: 0.9012 - val_loss: 0.3773


In [72]:
model.save("sentiment_bilstm_model.h5")

In [ ]:
""